# Eksperimen Machine Learning - Heart Disease Dataset

**Nama:** ardir  
**Kelas:** Membangun Sistem Machine Learning  
**Dataset:** Heart Disease (Cleveland) - UCI ML Repository  
**Tanggal:** 2026-05-19

---

## Daftar Isi
1. Data Loading
2. Exploratory Data Analysis (EDA)
3. Data Preprocessing
4. Modeling (Eksperimen Awal)
5. Evaluasi Model
6. Kesimpulan

## 1. Data Loading

Pada tahap ini, kita akan memuat dataset Heart Disease (Cleveland) dari UCI ML Repository. Dataset ini berisi data klinis pasien untuk prediksi penyakit jantung.

In [ ]:
# Import library yang diperlukan
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
import warnings
warnings.filterwarnings("ignore")

print("Library berhasil dimuat!")

In [ ]:
# Load dataset dari UCI ML Repository
UCI_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

column_names = [
    "age", "sex", "cp", "trestbps", "chol", "fbs",
    "restecg", "thalach", "exang", "oldpeak", "slope",
    "ca", "thal", "target"
]

df = pd.read_csv(UCI_URL, names=column_names, na_values="?")
df["target"] = (df["target"] > 0).astype(int)

print(f"Total Dataset: {len(df)} baris, {df.shape[1]} kolom")
print(f"No Disease: {(df['target']==0).sum()}, Disease: {(df['target']==1).sum()}")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Exploratory Data Analysis (EDA)

Pada tahap ini kita akan mengeksplorasi distribusi data, korelasi antar fitur, dan keseimbangan kelas.

In [ ]:
# Cek missing values
print("Missing Values:")
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

In [ ]:
print(f"Jumlah baris duplikat: {df.duplicated().sum()}")

In [ ]:
# Distribusi target
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
target_counts = df["target"].value_counts().sort_index()
colors = ["#2196F3", "#FF5722"]
axes[0].bar(["No Disease (0)", "Disease (1)"], target_counts.values, color=colors)
axes[0].set_xlabel("Target")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribusi Target Heart Disease")
axes[1].pie(target_counts, labels=["No Disease", "Disease"], autopct="%1.1f%%",
            colors=colors, startangle=90)
axes[1].set_title("Proporsi No Disease vs Disease")
plt.tight_layout()
plt.show()

In [ ]:
# Distribusi fitur numerik
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
fig, axes = plt.subplots(4, 4, figsize=(16, 14))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    if i < len(axes):
        axes[i].hist(df[col].dropna(), bins=30, color="#3498DB", alpha=0.7, edgecolor="black")
        axes[i].set_title(col, fontsize=10)
        axes[i].set_ylabel("Frequency")
for j in range(len(num_cols), len(axes)):
    fig.delaxes(axes[j])
plt.suptitle("Distribusi Fitur Numerik", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 10))
corr_matrix = df[num_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f",
            cmap="coolwarm", center=0, linewidths=0.5,
            square=True, cbar_kws={"shrink": 0.8})
plt.title("Korelasi Antar Fitur", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot per target
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
important_features = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]
for i, col in enumerate(important_features):
    sns.boxplot(data=df, x="target", y=col, ax=axes[i], palette="viridis")
    axes[i].set_title(f"{col} vs Target")
plt.suptitle("Fitur Penting vs Target", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Data Preprocessing

Tahapan preprocessing meliputi:
- Handling missing values & duplicates
- Feature engineering
- Scaling fitur numerik
- Train-validation-test split

In [ ]:
# 3.1 Handle missing values & duplikasi
print(f"Sebelum: {len(df)} baris")
df_clean = df.copy()
for col in df_clean.columns:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].median(), inplace=True)
df_clean = df_clean.drop_duplicates()
print(f"Sesudah: {len(df_clean)} baris")

In [ ]:
# 3.2 Feature Engineering
df_clean = df_clean.copy()
df_clean["heart_rate_reserve"] = df_clean["thalach"] / (220 - df_clean["age"]).replace(0, 1)
df_clean["cholesterol_age_ratio"] = df_clean["chol"] / df_clean["age"].replace(0, 1)
df_clean["bp_chol_interaction"] = df_clean["trestbps"] * df_clean["chol"] / 10000
df_clean["exercise_risk"] = df_clean["oldpeak"] * (df_clean["exang"] + 1)
df_clean["age_sex_risk"] = df_clean["age"] * (1 + df_clean["sex"] * 0.2)

print(f"Total fitur: {df_clean.shape[1]}")
print(f"Distribusi target:")
print(df_clean["target"].value_counts())

In [ ]:
# 3.3 Definisikan fitur dan target
X = df_clean.drop(columns=["target"])
y = df_clean["target"]
print(f"Fitur: {X.shape}")
print(f"Target: {y.shape}")
print(f"Nama fitur: {X.columns.tolist()}")

In [ ]:
# 3.4 Train-Validation-Test Split
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.125, random_state=42, stratify=y_temp)
total = len(X)
print(f"Train: {X_train.shape[0]} samples ({X_train.shape[0]/total*100:.1f}%)")
print(f"Val:   {X_val.shape[0]} samples ({X_val.shape[0]/total*100:.1f}%)")
print(f"Test:  {X_test.shape[0]} samples ({X_test.shape[0]/total*100:.1f}%)")

In [ ]:
# 3.5 Scaling
scaler = StandardScaler()
num_cols_scale = X_train.select_dtypes(include=[np.number]).columns.tolist()
X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()
X_train_scaled[num_cols_scale] = scaler.fit_transform(X_train[num_cols_scale])
X_val_scaled[num_cols_scale] = scaler.transform(X_val[num_cols_scale])
X_test_scaled[num_cols_scale] = scaler.transform(X_test[num_cols_scale])
print("Scaling selesai!")
X_train_scaled.head()

## 4. Modeling (Eksperimen Awal)

Kita akan mencoba beberapa model:
1. Random Forest Classifier
2. Gradient Boosting Classifier
3. Logistic Regression

In [ ]:
# 4.1 Random Forest
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)
rf_train_acc = accuracy_score(y_train, rf_model.predict(X_train_scaled))
rf_val_acc = accuracy_score(y_val, rf_model.predict(X_val_scaled))
rf_test_acc = accuracy_score(y_test, rf_model.predict(X_test_scaled))
print("Random Forest Results:")
print(f"  Train Accuracy: {rf_train_acc:.4f}")
print(f"  Val Accuracy:   {rf_val_acc:.4f}")
print(f"  Test Accuracy:  {rf_test_acc:.4f}")

In [ ]:
# 4.2 Gradient Boosting
gb_model = GradientBoostingClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
gb_model.fit(X_train_scaled, y_train)
gb_train_acc = accuracy_score(y_train, gb_model.predict(X_train_scaled))
gb_val_acc = accuracy_score(y_val, gb_model.predict(X_val_scaled))
gb_test_acc = accuracy_score(y_test, gb_model.predict(X_test_scaled))
print("Gradient Boosting Results:")
print(f"  Train Accuracy: {gb_train_acc:.4f}")
print(f"  Val Accuracy:   {gb_val_acc:.4f}")
print(f"  Test Accuracy:  {gb_test_acc:.4f}")

In [ ]:
# 4.3 Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_train_acc = accuracy_score(y_train, lr_model.predict(X_train_scaled))
lr_val_acc = accuracy_score(y_val, lr_model.predict(X_val_scaled))
lr_test_acc = accuracy_score(y_test, lr_model.predict(X_test_scaled))
print("Logistic Regression Results:")
print(f"  Train Accuracy: {lr_train_acc:.4f}")
print(f"  Val Accuracy:   {lr_val_acc:.4f}")
print(f"  Test Accuracy:  {lr_test_acc:.4f}")

## 5. Evaluasi Model

Membandingkan performa semua model dan mengevaluasi model terbaik secara detail.

In [ ]:
# 5.1 Perbandingan Model
results = pd.DataFrame({
    "Model": ["Random Forest", "Gradient Boosting", "Logistic Regression"],
    "Train Acc": [rf_train_acc, gb_train_acc, lr_train_acc],
    "Val Acc": [rf_val_acc, gb_val_acc, lr_val_acc],
    "Test Acc": [rf_test_acc, gb_test_acc, lr_test_acc]
})
print("Perbandingan Performa Model:")
print(results.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(results))
width = 0.25
bars1 = ax.bar(x - width, results["Train Acc"], width, label="Train", color="#2196F3")
bars2 = ax.bar(x, results["Val Acc"], width, label="Validation", color="#4CAF50")
bars3 = ax.bar(x + width, results["Test Acc"], width, label="Test", color="#FF5722")
ax.set_ylabel("Accuracy")
ax.set_title("Perbandingan Performa Model", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(results["Model"])
ax.legend()
ax.set_ylim(0, 1.1)
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f"{height:.3f}", xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords="offset points", ha="center", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# 5.2 Classification Report
y_pred_test = rf_model.predict(X_test_scaled)
print("Classification Report - Random Forest (Test Set):")
print(classification_report(y_test, y_pred_test, target_names=["no_disease", "disease"]))

In [ ]:
# 5.3 Confusion Matrix
cm = confusion_matrix(y_test, y_pred_test)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["no_disease", "disease"], yticklabels=["no_disease", "disease"])
plt.title("Confusion Matrix - Random Forest", fontsize=14, fontweight="bold")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
# 5.4 Feature Importance
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]
plt.figure(figsize=(12, 6))
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(indices)))
plt.barh(range(len(indices)), importances[indices][::-1], color=colors)
plt.yticks(range(len(indices)), [X_train_scaled.columns[i] for i in indices[::-1]])
plt.xlabel("Importance")
plt.title("Feature Importances - Random Forest", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Kesimpulan

### Temuan Utama:
1. **Dataset**: Heart Disease (Cleveland) dataset terdiri dari 303 sampel dengan 13 fitur klinis.
2. **EDA**: Distribusi target cukup seimbang. Fitur `cp`, `thalach`, `ca`, dan `oldpeak` memiliki korelasi kuat dengan target.
3. **Preprocessing**: Dilakukan feature engineering (heart_rate_reserve, cholesterol_age_ratio, bp_chol_interaction, exercise_risk, age_sex_risk), handling missing values, dan scaling.
4. **Model Terbaik**: Random Forest Classifier memberikan performa terbaik.

### Langkah Selanjutnya:
- Implementasi automated preprocessing dengan `automate_ardir.py`
- Integrasi dengan MLflow untuk tracking eksperimen
- Hyperparameter tuning lebih lanjut
- Deployment model menggunakan FastAPI
- Monitoring dengan Prometheus dan Grafana